# Quran Qari (Reciter) Classifier

**Goal.** Given a short audio clip of Quran recitation, predict which qari (reciter) is reading. Shazam-style — user uploads audio, model returns top matches with confidence.

**Dataset.** Assembled from three sources into a single unified format:
- 12 Arab qaris from Kaggle `mohammedalrajeh/quran-recitations-for-audio-classification` (pre-segmented WAV)
- 15 extra Arab qaris scraped from everyayah.com (Juz 30 sample per qari)
- 3–5 Uzbek qaris (added manually via `data/raw_uzbek/` — YouTube blocks yt-dlp without cookies)

All audio unified: 5-second clips, mono, 16 kHz, peak-normalized. Features: MFCC + delta + delta-delta + spectral stats.

**Pipeline.**
1. Setup
2. Load features + labels
3. EDA (class balance, feature distributions, PCA)
4. Train / test split (stratified)
5. Baseline — Logistic Regression
6. Bake-off (LogReg, KNN, RF, XGBoost)
7. Hyper-parameter tuning
8. Final evaluation (per-class report, confusion matrix, top-3 accuracy)
9. Feature importance
10. Save artifacts for deployment
11. Streamlit deployment notes

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    top_k_accuracy_score, ConfusionMatrixDisplay,
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42
DATA_DIR = Path('data')

## 2. Load features & labels

Features + labels are produced by `scripts/segment_and_features.py`. Run that first if `data/features.parquet` is missing.

In [ ]:
features_path = DATA_DIR / 'features.parquet'
labels_path   = DATA_DIR / 'labels.csv'

assert features_path.exists(), f'{features_path} missing — run scripts/segment_and_features.py first'
assert labels_path.exists(),   f'{labels_path} missing — run scripts/segment_and_features.py first'

feat_df  = pd.read_parquet(features_path)
label_df = pd.read_csv(labels_path)
df = feat_df.merge(label_df, on='clip')
print('Shape:', df.shape)
print('Reciters:', df['reciter'].nunique())
df.head()

## 3. Exploratory Data Analysis

### 3.1 Class distribution — clips per qari

In [ ]:
vc = df['reciter'].value_counts().sort_values(ascending=True)
print(vc)
print(f'\nTotal clips: {len(df)}')
print(f'Min per class: {vc.min()}  |  Max per class: {vc.max()}  |  Median: {int(vc.median())}')

plt.figure(figsize=(10, max(4, len(vc) * 0.3)))
vc.plot(kind='barh', color='steelblue')
plt.title('Clips per qari')
plt.xlabel('# clips')
plt.tight_layout()
plt.show()

### 3.2 Feature space overview

Quick check: how many features per clip, what are their scales?

In [ ]:
feature_cols = [c for c in df.columns if c not in ('clip', 'reciter')]
print(f'Features per clip: {len(feature_cols)}')
print(df[feature_cols].describe().T[['mean', 'std', 'min', 'max']].round(2).head(10))

### 3.3 PCA — 2D projection colored by qari

If qaris cluster visibly in 2D, classifier will thrive. If they overlap heavily, need richer features (CNN on spectrograms).

In [ ]:
scaler_diag = StandardScaler()
X_diag = scaler_diag.fit_transform(df[feature_cols])
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca.fit_transform(X_diag)

plt.figure(figsize=(11, 8))
reciters = sorted(df['reciter'].unique())
palette = sns.color_palette('tab20', len(reciters))
for r, color in zip(reciters, palette):
    mask = df['reciter'].values == r
    plt.scatter(Z[mask, 0], Z[mask, 1], s=8, alpha=0.5, color=color, label=r)
plt.title(f'PCA — first 2 components ({pca.explained_variance_ratio_.sum()*100:.1f}% variance)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Feature preparation & stratified split

In [ ]:
X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df['reciter'])
print('Classes:', list(le.classes_))
print('X:', X.shape, '| y:', y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## 5. Baseline — Logistic Regression

In [ ]:
baseline = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, n_jobs=-1)
baseline.fit(X_train_s, y_train)
pred  = baseline.predict(X_test_s)
proba = baseline.predict_proba(X_test_s)

print(f'Accuracy       : {accuracy_score(y_test, pred):.4f}')
print(f'Top-3 accuracy : {top_k_accuracy_score(y_test, proba, k=3):.4f}')
print(f'\nClassification report:')
print(classification_report(y_test, pred, target_names=le.classes_))

## 6. Model bake-off — defaults

Same split, uniform metrics. Track accuracy, top-3, macro-F1.

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, n_jobs=-1),
    'KNN':                KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'RandomForest':       RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
}
if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=400, learning_rate=0.1, max_depth=5,
        eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1,
    )

rows = []
for name, mdl in models.items():
    mdl.fit(X_train_s, y_train)
    pred  = mdl.predict(X_test_s)
    proba = mdl.predict_proba(X_test_s)
    rep   = classification_report(y_test, pred, output_dict=True, zero_division=0)
    rows.append({
        'model':    name,
        'accuracy': accuracy_score(y_test, pred),
        'top3':     top_k_accuracy_score(y_test, proba, k=3),
        'macro_f1': rep['macro avg']['f1-score'],
    })

results_df = pd.DataFrame(rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
print(results_df.round(4).to_string(index=False))

results_df.set_index('model')[['accuracy', 'top3', 'macro_f1']].plot(kind='bar', figsize=(10, 5))
plt.title('Bake-off — defaults')
plt.ylabel('score')
plt.xticks(rotation=25)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Hyper-parameter tuning — top model

Tune the leader by macro-F1 with 5-fold CV. Others typically don't catch up on tabular MFCC — tuning is a spot-check, not a horse race.

In [ ]:
top_name = results_df.iloc[0]['model']
print(f'Tuning: {top_name}')

if top_name == 'RandomForest':
    est = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    grid = {'n_estimators': [300, 600], 'max_depth': [None, 20, 40], 'min_samples_split': [2, 5]}
elif top_name == 'XGBoost' and HAS_XGB:
    est = XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1)
    grid = {'n_estimators': [300, 600], 'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1]}
elif top_name == 'LogisticRegression':
    est = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, n_jobs=-1)
    grid = {'C': [0.1, 1, 10], 'penalty': ['l2']}
elif top_name == 'KNN':
    est = KNeighborsClassifier(n_jobs=-1)
    grid = {'n_neighbors': [3, 5, 11, 21], 'weights': ['uniform', 'distance']}

gs = GridSearchCV(est, grid, scoring='f1_macro', cv=5, n_jobs=-1, verbose=1)
gs.fit(X_train_s, y_train)
print(f'best CV macro-F1 : {gs.best_score_:.4f}')
print(f'best params      : {gs.best_params_}')
best_model = gs.best_estimator_

## 8. Final evaluation

In [ ]:
pred  = best_model.predict(X_test_s)
proba = best_model.predict_proba(X_test_s)

print(f'Accuracy       : {accuracy_score(y_test, pred):.4f}')
print(f'Top-3 accuracy : {top_k_accuracy_score(y_test, proba, k=3):.4f}')
print(f'\nClassification report:')
print(classification_report(y_test, pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(max(8, len(le.classes_) * 0.5), max(6, len(le.classes_) * 0.5)))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title('Confusion matrix — tuned best model')
plt.tight_layout()
plt.show()

## 9. Feature importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(25)
    imp[::-1].plot(kind='barh', figsize=(9, 8), color='steelblue')
    plt.title('Top-25 features')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_'):
    imp = pd.Series(np.abs(best_model.coef_).mean(axis=0), index=feature_cols).sort_values(ascending=False).head(25)
    imp[::-1].plot(kind='barh', figsize=(9, 8), color='steelblue')
    plt.title('Top-25 features (mean |coef| across classes)')
    plt.tight_layout()
    plt.show()
else:
    print(f'{type(best_model).__name__} has no direct feature-importance attribute.')

## 10. Save artifacts for deployment

In [ ]:
artifacts = {
    'model':         best_model,
    'scaler':        scaler,
    'label_encoder': le,
    'feature_cols':  feature_cols,
    'sr':            16000,
    'clip_sec':      5,
    'model_name':    type(best_model).__name__,
}
joblib.dump(artifacts, 'qari_artifacts.joblib')
print('Saved: qari_artifacts.joblib')

## 11. Streamlit deployment

See `app.py` — Shazam-style upload:
1. User uploads an audio file (mp3 / wav / m4a).
2. App loads, converts to 16 kHz mono, segments into 5s clips.
3. Extracts MFCC + delta + spectral features per clip using the same pipeline.
4. Averages per-clip probabilities.
5. Displays top-3 predicted qaris with confidence bars + waveform + mel spectrogram.

**Run locally:**
```bash
streamlit run app.py
```

**Deploy to Streamlit Cloud:**
1. Push `app.py`, `qari_artifacts.joblib`, `requirements.txt` to a GitHub repo.
2. Go to https://share.streamlit.io → New app → pick repo → Deploy.
3. Public URL is permanent.